# 🇸🇦 Saudi Digital Talent Insights
### Analyzing the growth of the digital workforce in Saudi Arabia

Hi, I'm **Rimas Alshahrani**, a Data Analyst passionate about turning data into decisions that matter.

**Contact:** +966538108040 | https://www.linkedin.com/in/rimas-abdulrhman-a0018a3b6/
**Date:** October 2026

---

## 🎯 Purpose
This project analyzes the Saudi digital and tech workforce to support data-driven decisions in talent development, in line with the Ministry of Communications and Information Technology (MCIT) priorities and Saudi Vision 2030.

## ❓ Key Questions
1. How has the digital workforce grown over recent years?
2. What is the participation rate of women in tech roles?
3. How is digital talent distributed across regions and fields?
4. Where are the gaps, and what can be done to close them?

## 📂 Data Sources
- General Authority for Statistics (GASTAT)
- Saudi Open Data Portal (open.data.gov.sa)

## 🛠 Tools
Python · Pandas · Plotly · Streamlit

## 🗂 Project Structure
1. Data Collection & Cleaning
2. Exploratory Data Analysis (EDA)
3. Interactive Dashboard
4. Insights & Recommendations


In [1]:
# ==============================================
# 1. Data Collection: Merge quarterly GASTAT files
# ==============================================
import pandas as pd
import re
from google.colab import files

# Upload all quarterly Excel files at once
uploaded = files.upload()

columns = [
    "Saudi_Male", "Saudi_Female", "Saudi_Total",
    "NonSaudi_Male", "NonSaudi_Female", "NonSaudi_Total",
    "All_Male", "All_Female", "All_Total"
]


Saving Labor Market Statistics Q2 2025 EN.xlsx to Labor Market Statistics Q2 2025 EN.xlsx
Saving Labor Market Statistics Q2 2026 EN (2).xlsx to Labor Market Statistics Q2 2026 EN (2).xlsx
Saving LM tables_Q2_2022_EN_(%).xlsx to LM tables_Q2_2022_EN_(%).xlsx
Saving LM tables_Q2_2023_EN(%).xlsx to LM tables_Q2_2023_EN(%).xlsx
Saving LM tables_Q2_2024_EN.xlsx to LM tables_Q2_2024_EN.xlsx


In [2]:

rows = []
for file_name in uploaded.keys():
    # Get year and quarter from the file name (e.g. Q2_2026)
    year = re.search(r"20\d{2}", file_name).group()
    quarter = re.search(r"Q\d", file_name, re.IGNORECASE).group().upper()

    # Table 3-5: employed by main economic activity (%)
    sheet = pd.read_excel(file_name, sheet_name="3-5", header=None)

    # Find the Information and Communication row
    ict_row = sheet[sheet[0].astype(str).str.contains("Information and Communication", case=False)]
    values = ict_row.iloc[0, 1:10].astype(float).tolist()

    rows.append([f"{year} {quarter}", int(year), quarter] + values)

ict = pd.DataFrame(rows, columns=["Period", "Year", "Quarter"] + columns)
ict = ict.sort_values(["Year", "Quarter"]).reset_index(drop=True)

# Save as one clean file
ict.to_csv("ict_workforce.csv", index=False)
ict

,Period,Year,Quarter,Saudi_Male,Saudi_Female,Saudi_Total,NonSaudi_Male,NonSaudi_Female,NonSaudi_Total,All_Male,All_Female,All_Total
0,2022 Q2,2022,Q2,1.061526,1.430929,1.175867,1.181001,0.298252,1.110900,1.144343,1.087728,1.135011
1,2023 Q2,2023,Q2,1.218414,1.188013,1.208929,1.312494,0.221065,1.227260,1.284800,0.888940,1.220687
2,2024 Q2,2024,Q2,1.176941,1.097679,1.150887,1.240739,0.239583,1.170011,1.222806,0.853800,1.163295
3,2025 Q2,2025,Q2,1.274397,0.963385,1.169942,1.328089,0.207690,1.228048,1.314128,0.694685,1.209153
4,2026 Q2,2026,Q2,1.228620,1.097391,1.184483,1.468728,0.315970,1.370327,1.409028,0.818006,1.312118


Exploratory Data Analysis (EDA)

In [3]:
# Table 3-1: share of men and women among all employed (time series)
latest_file = [f for f in uploaded if "2026" in f][0]
gender = pd.read_excel(latest_file, sheet_name="3-1", header=None)

# Keep only rows like "2022 Q2", and the female share columns
gender = gender[gender[0].astype(str).str.match(r"^20\d{2} Q\d")].iloc[:, [0, 2, 8]]
gender.columns = ["Period", "Saudi_FemaleShare", "All_FemaleShare"]
gender[["Saudi_FemaleShare", "All_FemaleShare"]] = gender[["Saudi_FemaleShare", "All_FemaleShare"]].astype(float)

df = ict.merge(gender, on="Period")

# Women's share inside the ICT workforce
df["Saudi_Women_in_ICT"] = df["Saudi_FemaleShare"] * df["Saudi_Female"] / df["Saudi_Total"]
df["All_Women_in_ICT"]   = df["All_FemaleShare"]   * df["All_Female"]   / df["All_Total"]

df[["Period", "All_Total", "Saudi_Women_in_ICT", "All_Women_in_ICT"]].round(1)

,Period,All_Total,Saudi_Women_in_ICT,All_Women_in_ICT
0,2022 Q2,1.1,37.7,15.8
1,2023 Q2,1.2,30.7,11.8
2,2024 Q2,1.2,31.4,11.8
3,2025 Q2,1.2,27.7,9.7
4,2026 Q2,1.3,31.2,10.2


In [4]:
import plotly.express as px

# 1. ICT share of total employment
fig1 = px.line(df, x="Period", y="All_Total", markers=True,
               title="ICT Sector Share of Total Employment (%)",
               labels={"All_Total": "% of employed"})
fig1.show()

# 2. ICT share by group
long = df.melt(id_vars="Period",
               value_vars=["Saudi_Male", "Saudi_Female", "NonSaudi_Total"],
               var_name="Group", value_name="Share")
fig2 = px.bar(long, x="Period", y="Share", color="Group", barmode="group",
              title="Share Working in ICT by Group (%)")
fig2.show()

# 3. Women in the ICT workforce vs MCIT benchmark
fig3 = px.line(df, x="Period", y=["Saudi_Women_in_ICT", "All_Women_in_ICT"], markers=True,
               title="Women's Share of the ICT Workforce (%)",
               labels={"value": "%", "variable": ""})
fig3.add_hline(y=35, line_dash="dash", annotation_text="MCIT reported: 35%")
fig3.show()

In [5]:
df.round(3).to_csv("ict_dashboard.csv", index=False)

## 📊 Key Findings
1. [Finding 1 with a number, e.g. "The digital workforce grew by X% between 20XX and 20XX"]
2. [Finding 2]
3. [Finding 3]

## 💡 Recommendations
1. [Recommendation linked to Finding 1]
2. [Recommendation linked to Finding 2]
3. [Recommendation linked to Finding 3]

## 🔭 Next Steps
- Add more recent data as it becomes available
- Expand the analysis to include [e.g. salaries, specializations, training programs]

---

## 🔗 Links
- **Live Dashboard:** [Streamlit link]
- **GitHub Repository:** [GitHub link]

Thank you for reading! I'd be happy to hear your feedback.
**Rimas Alshahrani** | [LinkedIn link]
